**Note: For this assignment, you may only use standard Python and the `re` (Regular Expression) module. Advanced libraries such as NumPy, Pandas are not permitted**

## Exercises 1

Use `re.search` to find whether a string contains a phone number. The pattern that you write should detect a phone number in the following strings.  
```
"Call me at 382-384-3840."  
"my number is (510) 849-3519. Call me!"
```  
And not find a match in the following strings. 
```
"my number is 510-849-35192"  
"here’s my number: 510-849.3519"
``` 
Consider making your own tests as well  

In [1]:
import re

In [2]:
# YOUR CODE HERE
phone_pattern = r'(?:^|\D)(?:\(\d{3}\) \d{3}-\d{4}|\d{3}-\d{3}-\d{4})(?:$|\D)'

test_strings = [
    "Call me at 382-384-3840.",
    "my number is (510) 849-3519. Call me!",
    "my number is 510-849-35192",
    "here’s my number: 510-849.3519"
]

for s in test_strings:
    match = re.search(phone_pattern, s)
    print(f"'{s}' ==> {'Found' if match else 'Not found'}")

'Call me at 382-384-3840.' ==> Found
'my number is (510) 849-3519. Call me!' ==> Found
'my number is 510-849-35192' ==> Not found
'here’s my number: 510-849.3519' ==> Not found


## Exercise 2

Use `re.sub` to alter the string below so that the dates have a common format that uses a dash for the day, month, and year separator.  
```
03/12/2018, 03.13.18, 03/14/2018, 03:15:2018
```

In [3]:
# YOUR CODE HERE
date_string = "03/12/2018, 03.13.18, 03/14/2018, 03:15:2018"

step1 = re.sub(r'[-.:]', '/', date_string)

def format_date(m):
    month, day, year = m.groups()
    if len(year) == 2:
        year = '20' + year
    return f"{day.zfill(2)}-{month.zfill(2)}-{year}"


s = re.sub(r'[-.:]', '/', date_string)
result = re.sub(r'\b(\d{1,2})/(\d{1,2})/(\d{2}|\d{4})\b', format_date, s)
print(result)

12-03-2018, 13-03-2018, 14-03-2018, 15-03-2018


## Exercise 3

Consider the first five sentences of the novel “Little Women” below. Extract the spoken dialog from each sentence.

In [4]:
text = '''
"Christmas won't be Christmas without any presents," grumbled Jo, lying on the rug.
"It's so dreadful to be poor!" sighed Meg, looking down at her old dress.
"I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all," added little Amy, with an injured sniff.
"We've got Father and Mother, and each other," said Beth contentedly from her corner.
The four young faces on which the firelight shone brightened at the cheerful words, but darkened again as Jo said sadly, "We haven't got Father, and shall not have him for a long time."
'''
print(text)


"Christmas won't be Christmas without any presents," grumbled Jo, lying on the rug.
"It's so dreadful to be poor!" sighed Meg, looking down at her old dress.
"I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all," added little Amy, with an injured sniff.
"We've got Father and Mother, and each other," said Beth contentedly from her corner.
The four young faces on which the firelight shone brightened at the cheerful words, but darkened again as Jo said sadly, "We haven't got Father, and shall not have him for a long time."



In [5]:
# YOUR CODE HERE
clean_dialogs = [
    re.sub(r'[.!?;,:]+$', '', d.strip())
    for d in re.findall(r'"([^"]*)"', text)
]
for dialog in clean_dialogs:
    print(dialog)

Christmas won't be Christmas without any presents
It's so dreadful to be poor
I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all
We've got Father and Mother, and each other
We haven't got Father, and shall not have him for a long time


## Exercise 4

In this exercise, you you working with ```email_test.txt``` file (attached), using Regular Expression.\
`Original Dataset: https://www.kaggle.com/datasets/rtatman/fraudulent-email-corpus`

In [6]:
# YOUR CODE HERE
with open('email_test.txt', 'r', encoding='utf-8', errors='ignore') as f:
    content = f.read()

#Split emails: start with "From r " and end before the next email
email_blocks = re.findall(r'(^From r  .*?(?=\nFrom r  |\Z))', content, re.MULTILINE | re.DOTALL)
emails = [block.strip() for block in email_blocks if block.strip()]

print(f"Total emails loaded: {len(emails)}\n")

Total emails loaded: 1330



#### Simple Fraudulent email detection

1. Count how many emails contain urgency-related words (URGENT, IMMEDIATELY, QUICK, ASSISTANCE, CONFIDENTIAL). 
Calculate what percentage of emails use these tactics.

In [7]:
# YOUR CODE HERE
# urgency_pattern = r'\b(URGENT|IMMEDIATELY|QUICK|ASSISTANCE|CONFIDENTIAL)\b'
urgency_pattern = r'(URGENT|IMMEDIATELY|QUICK|ASSISTANCE|CONFIDENTIAL)'
emails_with_urgency = 0

for email in emails:
    if re.search(urgency_pattern, email, re.IGNORECASE):
        emails_with_urgency += 1

percentage_urgency = (emails_with_urgency / len(emails)) * 100 if emails else 0
print(f"Emails with urgency words: {emails_with_urgency}")
print(f"Total emails loaded: {len(emails)}\n")
print(f"Percentage: {percentage_urgency:.2f}%\n")

Emails with urgency words: 1184
Total emails loaded: 1330

Percentage: 89.02%



2. Find all mentions of money amounts in the email bodies (e.g., `US$25M`, `$100,000.00`, `USD$31,000,000.00`). Calculate:
- Total number of money mentions across all emails
- The largest amount mentioned
- The smallest amount mentioned
- Average amount per email

In [8]:
# YOUR CODE HERE
#money_pattern = r'(?:US|USD)?\$?\s*(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)\s*(?:Million|M|Billion|B|USD|US\$|\$)?'
money_pattern = r'(?:(?<=\s)|(?<=^)|(?<=US)|(?<=\())\$?\s*(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)\s*(?:Million|M|Billion|B|USD|US\$|\$)?'

all_amounts = []

for email in emails:
    matches = re.findall(money_pattern, email, re.IGNORECASE)
    for match in matches:
        amount_str = match[0] if match[0] else match[1]
        if amount_str:
            amount = float(amount_str.replace(',', ''))
            context = email[max(0, email.find(amount_str)-20):email.find(amount_str)+20].lower()

            if re.search(r'\b(million|m)\b', context):
                amount *= 1_000_000
            elif re.search(r'\b(billion|b)\b', context):
                amount *= 1_000_000_000

            all_amounts.append(amount)

print(f"Total money mentions: {len(all_amounts)}")
if all_amounts:
    print(f"Largest amount: ${max(all_amounts):,.2f}")
    print(f"Smallest amount: ${min(all_amounts):,.2f}")
    print(f"Average amount per email: ${sum(all_amounts)/len(emails):,.2f}\n")
else:
    print("No money amounts found\n")

Total money mentions: 25265
Largest amount: $9,000,000,000.00
Smallest amount: $0.00
Average amount per email: $320,838,387.15



3. Extract all mentions of deaths or deceased persons (e.g., "late father", "died", "deceased", "death of").\
   What percentage of emails use death as part of their story?

In [9]:
# YOUR CODE HERE
def levenshtein_distance(s1, s2):
    """Calculate Levenshtein distance between two strings"""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

def is_similar_word(word, target_words, threshold=1):
    """Check if word is similar to any target word using Levenshtein distance"""
    word_lower = word.lower()
    for target in target_words:
        if len(word_lower) >= len(target) - 2 and len(word_lower) <= len(target) + 2:
            distance = levenshtein_distance(word_lower, target)
            if distance <= threshold:
                return True, target, distance
    return False, None, None

death_keywords = ['late father', 'died', 'deceased', 'death', 'demise', 'passed', 'departed', 
                  'assassination', 'assassinated', 'killed', 'murdered', 'dead', 'dying']

emails_with_death = 0
death_matches = []

for i, email in enumerate(emails):
    words = re.findall(r'\b\w+\b', email)
    
    found_death_words = set()
    for word in words:
        if len(word) >= 3:
            is_similar, matched_keyword, distance = is_similar_word(word, death_keywords, threshold=1)
            if is_similar:
                found_death_words.add(f"{word.lower()}==>{matched_keyword}(d={distance})")
    
    if found_death_words:
        emails_with_death += 1
        subject_match = re.search(r'Subject:\s*(.+)', email)
        subject = subject_match.group(1).strip()[:50] if subject_match else "No subject"
        death_matches.append((i+1, subject, found_death_words))

death_percentage = (emails_with_death / len(emails)) * 100
print(f"Emails with death mentions: {emails_with_death}")
print(f"Percentage: {death_percentage:.2f}%")

print(f"\nSample emails with death-related words (first 5):")
for email_num, subject, words in death_matches[:5]:
    print(f"  Email #{email_num}: {subject}")
    print(f"    Matched words: {', '.join(sorted(words))}")
print()

Emails with death mentions: 1231
Percentage: 92.56%

Sample emails with death-related words (first 5):
  Email #1: URGENT BUSINESS ASSISTANCE AND PARTNERSHIP
    Matched words: assassinated==>assassinated(d=0), assassination==>assassination(d=0), dear==>dead(d=1)
  Email #2: URGENT ASSISTANCE /RELATIONSHIP (P)
    Matched words: dear==>dead(d=1), death==>death(d=0), head==>dead(d=1)
  Email #3: GOOD DAY TO YOU
    Matched words: deal==>dead(d=1), death==>death(d=0), died==>died(d=0), latefather==>late father(d=1)
  Email #4: GOOD DAY TO YOU
    Matched words: deal==>dead(d=1), death==>death(d=0), died==>died(d=0), latefather==>late father(d=1)
  Email #5: I Need Your Assistance.
    Matched words: dear==>dead(d=1), death==>death(d=0), demise==>demise(d=0), head==>dead(d=1), willed==>killed(d=1)



4. Many emails mention percentage splits of money (e.g., "70% for us", "20% for you", "10% for expenses"). \
   Extract all percentage distributions and identify the most common split pattern offered to recipients.

   Example: most common split patterns:\
   70% - 20% - 10%: appears 15 times\
   75% - 20% - 5%: appears 12 times\
    60% - 30% - 10%: appears 8 times\
    80% - 15% - 5%: appears 6 times\
    55% - 30% - 10% - 5%: appears 4 times

In [10]:
# YOUR CODE HERE
# This code is not calulated with 100%, if it meets 100%, it is ignored
split_patterns = {}

percentage_pattern = r'\b(\d{1,3})%'

for email in emails:
    percentages = re.findall(r'(\d+)\s*%', email)
    if percentages:
        pct_list = sorted(list(set([int(p) for p in percentages if int(p) != 100])), reverse=True)
        if pct_list:
            pattern = ' - '.join([f"{p}%" for p in pct_list])
            split_patterns[pattern] = split_patterns.get(pattern, 0) + 1

sorted_patterns = sorted(split_patterns.items(), key=lambda x: x[1], reverse=True)

print("Most common split patterns:")
for i, (pattern, count) in enumerate(sorted_patterns[:5], 1):
    print(f"{i}. {pattern}: appears {count} times")
print()

Most common split patterns:
1. 60% - 30% - 10%: appears 103 times
2. 70% - 25% - 5%: appears 83 times
3. 20%: appears 65 times
4. 30%: appears 54 times
5. 70% - 20% - 10%: appears 53 times



5. Create a "scam score" for each email based on:\
    Urgency keywords (1 point each)\
    Money mentions (2 points each)\
    Percentages offered (1 point each)\
    Death mentions (1 point)\
    ALL CAPS usage (1 point if >20% of text)

    ***Rank the top 10 highest-scoring emails*** 

In [11]:
# YOUR CODE HERE
email_scores = []

for i, email in enumerate(emails):
    score = 0
    
    # Urgency keywords (1 point each)
    urgency_matches = len(re.findall(urgency_pattern, email, re.IGNORECASE))
    score += urgency_matches
    
    # Money mentions (2 points each)
    money_matches = len(re.findall(money_pattern, email, re.IGNORECASE))
    score += money_matches * 2
    
    # Percentages (1 point each)
    pct_matches = len(re.findall(percentage_pattern, email))
    score += pct_matches
    
    # Death mentions (1 point)
    words = re.findall(r'\b\w+\b', email)
    has_death = False
    for word in words:
        if len(word) >= 3:
            death_keywords_short = ['late father', 'died', 'deceased', 'death', 'demise', 'passed', 'departed', 
                                        'assassination', 'assassinated', 'killed', 'murdered', 'dead', 'dying']
            is_similar, _, _ = is_similar_word(word, death_keywords_short, threshold=1)
            if is_similar:
                has_death = True
                break
    if has_death:
        score += 1
    
    # ALL CAPS usage (1 point if >20%)
    caps_chars = len(re.findall(r'[A-Z]', email))
    total_chars = len(re.findall(r'[A-Za-z]', email))
    if total_chars > 0 and (caps_chars / total_chars) > 0.2:
        score += 1
    

    subject_match = re.search(r'Subject:\s*(.+)', email)
    subject = subject_match.group(1).strip()[:50] if subject_match else "No subject"
    
    email_scores.append((i+1, score, subject))

email_scores.sort(key=lambda x: x[1], reverse=True)

print("Top 5 emails with highest scam scores:")
for rank, (email_num, score, subject) in enumerate(email_scores[:5], 1):
    print(f"{rank}. Email #{email_num} - Score: {score} - Subject: {subject}")
print()

Top 5 emails with highest scam scores:
1. Email #788 - Score: 855 - Subject: FROM  MONICA
2. Email #419 - Score: 255 - Subject: BUSINESS PROPOSER REQUESTING CHANGE OF OWNERSHIP
3. Email #417 - Score: 241 - Subject: PROJECT INTEREST.
4. Email #943 - Score: 186 - Subject: BOUNCE bg-misc@majordomo.cs.CU:    Non-member subm
5. Email #868 - Score: 157 - Subject: BUSINESS PROPOSAL , PARTNERSHIP REPLY URGENTLY



6. Identify emails that appear to be duplicates or near-duplicates (same sender, similar subject, sent within 24 hours). How many duplicate emails exist?

In [12]:
# YOUR CODE HERE
from datetime import datetime

# def levenshtein_distance(s1, s2): At Exercise 4.3
def subject_similarity(s1, s2):
    """Trả về % tương đồng giữa 2 subject (0-100)"""
    s1 = re.sub(r'\s+', ' ', s1.strip().lower())
    s2 = re.sub(r'\s+', ' ', s2.strip().lower())
    if not s1 and not s2:
        return 100.0
    if not s1 or not s2:
        return 0.0
    max_len = max(len(s1), len(s2))
    distance = levenshtein_distance(s1, s2)
    return (1 - distance / max_len) * 100

email_info = []
for idx, email in enumerate(emails):
    date_line = re.search(r'^From r  (.*)$', email, re.MULTILINE)
    date_obj = None
    date_str = ""
    if date_line:
        raw_date = date_line.group(1).strip()
        clean_date = re.sub(r'\s+[+-]\d{4}$', '', raw_date)
        try:
            date_obj = datetime.strptime(clean_date, "%a %b %d %H:%M:%S %Y")
            date_str = clean_date
        except:
            date_str = raw_date

    sender = re.search(r'^From:\s*(.+)$', email, re.MULTILINE)
    sender = sender.group(1).strip() if sender else ""

    subject = re.search(r'^Subject:\s*(.+)$', email, re.MULTILINE)
    subject = subject.group(1).strip() if subject else ""

    email_info.append({
        'id': idx + 1,
        'sender': sender,
        'subject': subject,
        'date': date_obj,
        'date_str': date_str
    })


parent = list(range(len(email_info)))

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py

# Combine emails based on criteria
for i in range(len(email_info)):
    for j in range(i + 1, len(email_info)):
        e1, e2 = email_info[i], email_info[j]

        if e1['sender'] != e2['sender']:
            continue

        if subject_similarity(e1['subject'], e2['subject']) < 90.0:
            continue

        if e1['date'] and e2['date']:
            if abs((e1['date'] - e2['date']).total_seconds()) > 86400:
                continue

        union(i, j)  # Combine into the same group


groups = {}
for i in range(len(email_info)):
    root = find(i)
    if root not in groups:
        groups[root] = []
    groups[root].append(i)

# Filter groups with ≥ 2 emails
duplicate_groups = [group for group in groups.values() if len(group) >= 2]
duplicate_groups = sorted(duplicate_groups, key=len, reverse=True)

print(f"Total duplicate groups found: {len(duplicate_groups)}")
print(f"Total emails in duplicate groups: {sum(len(g) for g in duplicate_groups)}\n")

print("Top 5 largest duplicate groups:")
for rank, group_indices in enumerate(duplicate_groups[:5], 1):
    emails_in_group = [email_info[i] for i in group_indices]
    sample = emails_in_group[0]

    print(f"\n{rank}. Group size: {len(emails_in_group)} emails")
    print(f"    Sender: {sample['sender']}")
    print(f"    Subject: {sample['subject']}")
    print(f"    Date range:")
    dates = [e['date_str'] for e in emails_in_group if e['date_str']]
    if dates:
        print(f"      From: {min(dates)}")
        print(f"      To:   {max(dates)}")
    else:
        print(f"      No date available")

    print(f"    Email IDs: {', '.join(str(email_info[i]['id']) for i in group_indices)}")

Total duplicate groups found: 197
Total emails in duplicate groups: 486

Top 5 largest duplicate groups:

1. Group size: 8 emails
    Sender: "Dr.David Clovis" <davidch@yahoo.com>
    Subject: Deal
    Date range:
      From: Sat Nov 20 13:21:36 2004
      To:   Sat Nov 20 13:31:41 2004
    Email IDs: 1261, 1262, 1263, 1264, 1265, 1266, 1267, 1268

2. Group size: 7 emails
    Sender: dradophilusjaja@tiscali.co.uk
    Subject: URGENTLY WAITING FOR YOUR REPLY
    Date range:
      From: Mon Sep 20 20:26:20 2004
      To:   Tue Sep 21 05:32:01 2004
    Email IDs: 1134, 1135, 1136, 1137, 1138, 1139, 1140

3. Group size: 6 emails
    Sender: "SESAY MASSAQUOE" <semassaq@o2.pl>
    Subject: I NEED YOUR HELP PLEASE
    Date range:
      From: Fri Jun  4 07:36:07 2004
      To:   Fri Jun  4 08:30:43 2004
    Email IDs: 909, 913, 914, 916, 917, 918

4. Group size: 6 emails
    Sender: 
    Subject: TESTAMENT
    Date range:
      From: Sun Dec  5 14:23:42 2004
      To:   Sun Dec  5 14:33:35 200